# 🚀 Análise Quantitativa Avançada e Backtest Institucional: BTCUSD
### Corretora: VT Markets | Motor Baseado no Robô Institucional PVSRA + ATR

Este notebook executa uma simulação quantitativa **barra-a-barra (livre de viés de antecipação)** com:
1. **Entrada na abertura da barra seguinte** após confirmação do fechamento do sinal.
2. **Custo real do spread da corretora** embutido em cada execução.
3. **Stop Loss dinâmico calibrado por volatilidade real (ATR 1.5x)**.
4. **Take Profit com Payoff Assimétrico (R:R 2.0:1)**.
5. **Filtro de Sazonalidade Institucional (Sessões Londres e Nova York)**.

In [ ]:
# 1. Instalação de dependências adicionais no Colab
!pip install --quiet pyarrow fastparquet plotly

In [ ]:
# 2. Carregamento dos dados históricos
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
except Exception:
    filename = 'data/BTCUSD_M15.parquet'

print(f"Carregando: {filename}")
if filename.endswith('.parquet'):
    df = pd.read_parquet(filename)
else:
    df = pd.read_csv(filename)

df['datetime'] = pd.to_datetime(df['datetime'])
df.sort_values('datetime', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total de barras carregadas: {len(df):,}")
print(f"Período: {df['datetime'].min()} até {df['datetime'].max()}")
df.head()

### 3. Engenharia de Features e Indicadores Institucionais (PVSRA + Médias + ATR)

In [ ]:
# Médias Móveis Exponenciais de Tendência
df['ema_fast'] = df['close'].ewm(span=50, adjust=False).mean()
df['ema_slow'] = df['close'].ewm(span=200, adjust=False).mean()

# ATR (Average True Range - 14 períodos)
prev_close = df['close'].shift(1)
tr1 = df['high'] - df['low']
tr2 = (df['high'] - prev_close).abs()
tr3 = (df['low'] - prev_close).abs()
df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
df['atr'] = df['tr'].rolling(14).mean()

# PVSRA (Price Volume Spread Analysis)
lookback = 10
df['avg_vol'] = df['tick_volume'].rolling(lookback).mean()
df['candle_spread'] = (df['close'] - df['open']).abs()
df['avg_spread'] = df['candle_spread'].rolling(lookback).mean()

cond_climax = (df['tick_volume'] >= 2.0 * df['avg_vol']) | ((df['tick_volume'] * df['candle_spread']) >= (2.0 * df['avg_vol'] * df['avg_spread']))
cond_high = df['tick_volume'] >= 1.5 * df['avg_vol']

df['pvsra'] = 0
df.loc[cond_high, 'pvsra'] = 1   # High Volume
df.loc[cond_climax, 'pvsra'] = 2 # Climax Volume

pvsra_labels = {0: 'NORMAL', 1: 'HIGH', 2: 'CLIMAX'}
print("Distribuição PVSRA:")
print(df['pvsra'].map(pvsra_labels).value_counts())

### 4. Motor de Execução de Ordens Realistas (Barra-a-Barra)

In [ ]:
def simulate_trading(df_data, rr_ratio=2.0, atr_multiplier=1.5, risk_percent=1.0, initial_balance=10000.0, session_filter=False):
    n = len(df_data)
    opens = df_data['open'].values
    highs = df_data['high'].values
    lows = df_data['low'].values
    closes = df_data['close'].values
    spreads = df_data['spread'].values * 0.01  # Spread em dólares no BTCUSD
    atrs = df_data['atr'].values
    pvsras = df_data['pvsra'].values
    ema_fasts = df_data['ema_fast'].values
    ema_slows = df_data['ema_slow'].values
    dates = df_data['datetime'].values
    hours = pd.to_datetime(dates).hour
    dows = pd.to_datetime(dates).dayofweek

    trades = []
    in_position = False
    pos_type = 0
    entry_price = 0.0
    sl_price = 0.0
    tp_price = 0.0
    entry_idx = 0
    trade_risk = 0.0
    balance = initial_balance
    equity_curve = [initial_balance]
    equity_dates = [dates[0]]

    for i in range(200, n - 1):
        if in_position:
            h, l = highs[i], lows[i]
            exit_trade = False
            exit_price = 0.0

            if pos_type == 1:
                if l <= sl_price:
                    exit_price = sl_price
                    exit_trade = True
                elif h >= tp_price:
                    exit_price = tp_price
                    exit_trade = True
            elif pos_type == -1:
                if h >= sl_price:
                    exit_price = sl_price
                    exit_trade = True
                elif l <= tp_price:
                    exit_price = tp_price
                    exit_trade = True

            if exit_trade:
                if pos_type == 1:
                    r = (exit_price - entry_price) / (entry_price - sl_price) if (entry_price - sl_price) > 0 else 0
                else:
                    r = (entry_price - exit_price) / (sl_price - entry_price) if (sl_price - entry_price) > 0 else 0
                pnl = r * trade_risk
                balance += pnl
                trades.append({
                    'entry_date': dates[entry_idx],
                    'exit_date': dates[i],
                    'type': 'BUY' if pos_type == 1 else 'SELL',
                    'entry_price': entry_price,
                    'exit_price': exit_price,
                    'pnl': pnl,
                    'r': r,
                    'balance': balance,
                    'duration_bars': i - entry_idx
                })
                in_position = False
                pos_type = 0

        equity_curve.append(balance)
        equity_dates.append(dates[i])

        if not in_position and not np.isnan(atrs[i]) and atrs[i] > 0:
            # Filtro de Sessão Institucional: Londres (8h-12h UTC) e NY (15h-19h UTC) em dias úteis
            if session_filter:
                is_london = (hours[i] >= 8) and (hours[i] < 12)
                is_ny = (hours[i] >= 15) and (hours[i] < 19)
                is_weekday = (dows[i] < 5)
                if not ((is_london or is_ny) and is_weekday):
                    continue

            c, o = closes[i], opens[i]
            pvsra = pvsras[i]
            trend_up = (c > ema_slows[i]) and (ema_fasts[i] > ema_slows[i])
            trend_down = (c < ema_slows[i]) and (ema_fasts[i] < ema_slows[i])
            vol_ok = (pvsra in [1, 2])

            buy_sig = trend_up and vol_ok and (c > o)
            sell_sig = trend_down and vol_ok and (c < o)

            if buy_sig or sell_sig:
                next_o = opens[i + 1]
                spread = spreads[i + 1]
                sl_dist = atrs[i] * atr_multiplier
                if sl_dist <= 0: continue

                pos_type = 1 if buy_sig else -1
                entry_idx = i + 1
                trade_risk = balance * (risk_percent / 100.0)

                if pos_type == 1:
                    entry_price = next_o + spread
                    sl_price = entry_price - sl_dist
                    tp_price = entry_price + (sl_dist * rr_ratio)
                else:
                    entry_price = next_o
                    sl_price = entry_price + sl_dist + spread
                    tp_price = entry_price - (sl_dist * rr_ratio)

                in_position = True

    return pd.DataFrame(trades), pd.DataFrame({'datetime': equity_dates, 'equity': equity_curve})

### 5. Comparativo: Sem Filtro vs. COM Filtro de Sessão Institucional

In [ ]:
def print_metrics(label, df_trades, df_equity, initial=10000.0):
    total = len(df_trades)
    wins = df_trades[df_trades['pnl'] > 0]
    wr = (len(wins) / total) * 100
    final = df_equity['equity'].iloc[-1]
    ret_pct = ((final - initial) / initial) * 100
    pf = wins['pnl'].sum() / abs(df_trades.loc[df_trades['pnl'] <= 0, 'pnl'].sum())
    
    df_equity['peak'] = df_equity['equity'].cummax()
    df_equity['dd'] = (df_equity['equity'] - df_equity['peak']) / df_equity['peak'] * 100
    mdd = abs(df_equity['dd'].min())
    
    print(f"=== {label} ===")
    print(f"Trades: {total:,} | Taxa de Acerto: {wr:.2f}%")
    print(f"Profit Factor: {pf:.2f} | Max Drawdown: {mdd:.2f}%")
    print(f"Retorno Líquido: {ret_pct:+.2f}% | Saldo Final: ${final:,.2f}\n")

# 1. Sem filtro de horário
trades_raw, eq_raw = simulate_trading(df, rr_ratio=2.0, session_filter=False)
print_metrics("PADRÃO: 24h/7d (Sem Filtro de Horário)", trades_raw, eq_raw)

# 2. Com filtro de sessões Londres + NY
trades_inst, eq_inst = simulate_trading(df, rr_ratio=2.0, session_filter=True)
print_metrics("OTIMIZADO: Sessões Institucionais (Londres 8-12h UTC / NY 15-19h UTC)", trades_inst, eq_inst)

### 6. Gráficos Comparativos da Curva de Capital e Drawdown

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3],
                    subplot_titles=('Evolução do Patrimônio ($10,000 Inicial)', 'Drawdown Subaquático (%)'))

# Equity Curves
fig.add_trace(go.Scatter(x=eq_raw['datetime'], y=eq_raw['equity'], line=dict(color='gray', dash='dash'), name='Sem Filtro (24/7)'), row=1, col=1)
fig.add_trace(go.Scatter(x=eq_inst['datetime'], y=eq_inst['equity'], line=dict(color='#00E676', width=2), name='Com Filtro Institucional (Londres/NY)'), row=1, col=1)

# Drawdowns
fig.add_trace(go.Scatter(x=eq_raw['datetime'], y=eq_raw['dd'], line=dict(color='gray', dash='dash'), name='DD Sem Filtro'), row=2, col=1)
fig.add_trace(go.Scatter(x=eq_inst['datetime'], y=eq_inst['dd'], fill='tozeroy', line=dict(color='#FF5252', width=1), name='DD Institucional'), row=2, col=1)

fig.update_layout(height=650, title='BTCUSD M15 - Comparativo de Performance Quantitativa', hovermode='x unified')
fig.show()